# 🔬 Soil GSD: Ridge Pipeline for Micro-Datasets

This notebook implements a robust **Linear Pipeline with L2 Regularization (Ridge)** specially tailored for wide micro-datasets ($p \gg n$). 

### Why LightGBM failed and why Ridge will shine:
With only 130 samples and over 500 deep features, tree-based models like LightGBM struggle to find statistically significant splits and default to predicting a flat global mean. 

By applying a `StandardScaler` combined with a multi-output `Ridge` regressor, we allow the model to evaluate all deep features simultaneously while strictly penalizing noise.

## Setup

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, "Training_labels_without_H374.csv")
PPM_PATH = os.path.join(DATA_DIR, "ppm.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "Training-All_Photos_without_H374")
TEST_IMG_DIR = os.path.join(DATA_DIR, "Test_All_Photos")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

Using device: cpu


## Mapping

In [2]:
def build_image_map(directory):
    img_map = {}
    if not os.path.exists(directory):
        return img_map
    for root, _, files in os.walk(directory):
        for file in files:
            name_idx = file.rfind('.')
            if name_idx != -1:
                sample_id = file[:name_idx].lower()
                img_map[sample_id] = os.path.join(root, file)
    return img_map

train_img_map = build_image_map(TRAIN_IMG_DIR)
test_img_map = build_image_map(TEST_IMG_DIR)

## Feature Extractor

In [3]:
# Feature extractor remains fixed, providing 512 structural embeddings
encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
feature_extractor = nn.Sequential(*list(encoder.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


## Feature Union Pipeline

In [4]:
def normalize_string(s):
    s = str(s).lower().strip()
    s = s.replace('ü', 'ue').replace('ö', 'oe').replace('ä', 'ae').replace('ß', 'ss')
    s = s.replace(',', '_').replace('-', '_').replace(' ', '_')
    while '__' in s:
        s = s.replace('__', '_')
    return s

def extract_hybrid_features(df, img_map, ppm_lookup, global_mean_ppm, set_name="Dataset"):
    features_list = []
    print(f"\n--- Processing & Feature Extraction for {set_name} ---")
    
    for idx, row in df.iterrows():
        sample_id = str(row["sample_id"])
        norm_sample_id = normalize_string(sample_id)
        
        matching_paths = []
        for key, path in img_map.items():
            norm_key = normalize_string(key)
            if (norm_sample_id in norm_key) or (norm_key in norm_sample_id) or (norm_sample_id.replace('hpc_', '') in norm_key):
                matching_paths.append(path)
                
        feat = {"sample_id": row["sample_id"]}
        
        if matching_paths:
            all_embs = []
            all_ppms = []
            all_ratios = []
            all_edges = []
            all_r_means = []
            all_g_means = []
            all_b_means = []
            
            for img_path in matching_paths:
                img = cv2.imread(img_path)
                if img is not None:
                    h, w = img.shape[0], img.shape[1]
                    all_ppms.append(ppm_lookup.get((w, h), global_mean_ppm))
                    all_ratios.append(w / h)
                    
                    # CV Features
                    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
                    edges = cv2.Canny(blurred, 30, 100)
                    all_edges.append(np.mean(edges > 0))
                    
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    all_r_means.append(np.mean(img_rgb[:, :, 0]))
                    all_g_means.append(np.mean(img_rgb[:, :, 1]))
                    all_b_means.append(np.mean(img_rgb[:, :, 2]))
                    
                    # Deep Features
                    pil_img = Image.fromarray(img_rgb)
                    tensor_img = img_transform(pil_img).unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        cnn_emb = feature_extractor(tensor_img).cpu().numpy().flatten()
                    all_embs.append(cnn_emb)
            
            if all_embs:
                feat["ppm"] = np.mean(all_ppms)
                feat["aspect_ratio"] = np.mean(all_ratios)
                feat["photo_count"] = len(all_embs)  # FIXED: Added to successful block
                feat["edge_density"] = np.mean(all_edges)
                feat["r_mean"] = np.mean(all_r_means)
                feat["g_mean"] = np.mean(all_g_means)
                feat["b_mean"] = np.mean(all_b_means)
                
                avg_emb = np.mean(all_embs, axis=0)
                for i, val in enumerate(avg_emb):
                    feat[f"cnn_{i}"] = val
            else:
                feat["ppm"] = global_mean_ppm
                feat["aspect_ratio"] = 1.33
                feat["photo_count"] = 0
                feat["edge_density"] = 0.0
                feat["r_mean"] = 127.0; feat["g_mean"] = 127.0; feat["b_mean"] = 127.0
        else:
            feat["ppm"] = global_mean_ppm
            feat["aspect_ratio"] = 1.33
            feat["photo_count"] = 0
            feat["edge_density"] = 0.0
            feat["r_mean"] = 127.0; feat["g_mean"] = 127.0; feat["b_mean"] = 127.0
            
        features_list.append(feat)
        
    return pd.DataFrame(features_list)

df_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
df_sub = pd.read_csv(SAMPLE_SUB_PATH)
df_ppm = pd.read_csv(PPM_PATH)

ppm_lookup = df_ppm.set_index(["width", "height"])["ppm"].to_dict()
global_mean_ppm = df_ppm["ppm"].mean()

X_train_df = extract_hybrid_features(df_train_labels, train_img_map, ppm_lookup, global_mean_ppm, "Train Set")
X_test_df = extract_hybrid_features(df_sub, test_img_map, ppm_lookup, global_mean_ppm, "Test Set")


--- Processing & Feature Extraction for Train Set ---

--- Processing & Feature Extraction for Test Set ---


## Scaled Ridge Multi-Output Training

In [5]:
target_cols = [col for col in df_train_labels.columns if col != "sample_id"]

# BULLETPROOF ALIGNMENT: Dynamically find columns that exist in BOTH train and test dataframes
common_cols = list(set(X_train_df.columns) & set(X_test_df.columns))
feature_cols = [col for col in common_cols if col != "sample_id"]

X = X_train_df[feature_cols].fillna(0)
Y = df_train_labels[target_cols].values
X_test = X_test_df[feature_cols].fillna(0)

print(f"➔ Aligned Feature Dimensions: {X.shape[1]} columns matched successfully.")

# Scaling + Ridge Regression with solid L2 penalty
base_ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge(alpha=50.0, random_state=42)
)
model = MultiOutputRegressor(base_ridge_pipeline)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(Y.shape)
test_preds = np.zeros((len(X_test), Y.shape[1]))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, Y)):
    X_tr, Y_tr = X.iloc[train_idx], Y[train_idx]
    X_va, Y_va = X.iloc[val_idx], Y[val_idx]
    
    model.fit(X_tr, Y_tr)
    
    val_pred = model.predict(X_va)
    oof_preds[val_idx] = val_pred
    test_preds += model.predict(X_test) / kf.n_splits
    
    print(f"Fold {fold+1} MAE: {mean_absolute_error(Y_va, val_pred):.4f}")

print(f"\n➔ Overall Out-of-Fold Grand MAE: {mean_absolute_error(Y, oof_preds):.4f}")

➔ Aligned Feature Dimensions: 519 columns matched successfully.
Fold 1 MAE: 8.9154
Fold 2 MAE: 8.7283
Fold 3 MAE: 10.6928
Fold 4 MAE: 11.3852
Fold 5 MAE: 11.0990

➔ Overall Out-of-Fold Grand MAE: 10.1641


## Post-Processing & Submission

In [6]:
# Enforce cumulative logic rules
final_test_preds = np.clip(test_preds, 0.0, 100.0)
final_test_preds = np.sort(final_test_preds, axis=1)

submission = pd.DataFrame(final_test_preds, columns=target_cols)
submission.insert(0, "sample_id", df_sub["sample_id"])

submission.to_csv("submission.csv", index=False)
print("\n➔ Clean submission.csv generated!")
display(submission.head())


➔ Clean submission.csv generated!


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,3.620156,8.396610,24.095805,62.788273,86.733017,90.245026,94.964580,100.000000,100.000000,100.000000,100.0
1,HPC_Audorfring,4.392373,5.809935,13.037225,39.543760,75.300848,85.404426,90.328457,100.000000,100.000000,100.000000,100.0
2,HPC_Muenster_BS6_9_0-10m,0.000000,0.000000,0.000000,0.000000,9.332237,19.092079,42.689930,73.962767,92.749974,99.683025,100.0
3,HPC_Airbus BS10-4bis7,6.574944,9.144111,19.788226,57.946698,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.0
4,HPC_Airbus BS12-5bis8,5.642241,8.836146,22.350543,67.198844,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.0
